# Lesson 14.5 - GenAI Observability and Evaluation Standards Playbook

## Learning Objectives
- Define a canonical telemetry schema for LLM/agent runs.
- Analyze synthetic traces for quality/cost/safety signals.
- Build standards-aligned release gate checks.

In [1]:
from __future__ import annotations

import pandas as pd
import numpy as np

## 1) Canonical GenAI Telemetry Fields (Practical Baseline)

In [2]:
core_fields = [
    "trace_id", "session_id", "model_id", "provider",
    "latency_ms", "prompt_tokens", "completion_tokens",
    "tool_calls", "safety_flag", "user_rating"
]
pd.DataFrame({"field": core_fields})

,field
0,trace_id
1,session_id
2,model_id
3,provider
4,latency_ms
5,prompt_tokens
6,completion_tokens
7,tool_calls
8,safety_flag
9,user_rating


## 2) Simulate Trace Dataset

In [3]:
np.random.seed(42)
rows = []
for i in range(300):
    rows.append({
        "trace_id": f"t_{i}",
        "session_id": f"s_{i//5}",
        "model_id": "example-llm-v2",
        "provider": "internal",
        "latency_ms": float(np.random.normal(850, 180)),
        "prompt_tokens": int(np.random.randint(120, 900)),
        "completion_tokens": int(np.random.randint(40, 300)),
        "tool_calls": int(np.random.poisson(1.2)),
        "safety_flag": int(np.random.rand() < 0.03),
        "user_rating": int(np.random.choice([1,2,3,4,5], p=[0.05,0.1,0.2,0.35,0.3])),
    })
traces = pd.DataFrame(rows)
traces.head()

,trace_id,session_id,model_id,provider,latency_ms,prompt_tokens,completion_tokens,tool_calls,safety_flag,user_rating
0,t_0,s_0,example-llm-v2,internal,939.408548,226,111,1,0,2
1,t_1,s_0,example-llm-v2,internal,825.112426,207,139,0,0,2
2,t_2,s_0,example-llm-v2,internal,891.957143,505,231,3,1,4
3,t_3,s_0,example-llm-v2,internal,871.239030,630,209,0,0,3
4,t_4,s_0,example-llm-v2,internal,889.974202,686,283,1,0,3


## 3) Standards-Oriented KPI Aggregation

In [4]:
kpi = {
    "p50_latency_ms": float(traces["latency_ms"].median()),
    "p95_latency_ms": float(traces["latency_ms"].quantile(0.95)),
    "avg_total_tokens": float((traces["prompt_tokens"] + traces["completion_tokens"]).mean()),
    "safety_flag_rate": float(traces["safety_flag"].mean()),
    "success_rate_rating_ge_4": float((traces["user_rating"] >= 4).mean()),
}
pd.DataFrame([kpi]).T.rename(columns={0: "value"})

,value
p50_latency_ms,843.137051
p95_latency_ms,1129.644792
avg_total_tokens,691.046667
safety_flag_rate,0.030000
success_rate_rating_ge_4,0.650000


## 4) Release Gate Example

In [5]:
release_gate = {
    "max_p95_latency_ms": 1300,
    "max_safety_flag_rate": 0.02,
    "min_success_rate": 0.70,
}

gate_result = {
    "latency_ok": kpi["p95_latency_ms"] <= release_gate["max_p95_latency_ms"],
    "safety_ok": kpi["safety_flag_rate"] <= release_gate["max_safety_flag_rate"],
    "quality_ok": kpi["success_rate_rating_ge_4"] >= release_gate["min_success_rate"],
}
gate_result["go_live"] = all(gate_result.values())
gate_result

{'latency_ok': True, 'safety_ok': False, 'quality_ok': False, 'go_live': False}

## 5) Mapping to OTel-Style Naming (Illustrative)

In [6]:
mapping = pd.DataFrame([
    {"internal": "model_id", "otel_like": "gen_ai.request.model"},
    {"internal": "latency_ms", "otel_like": "gen_ai.client.operation.duration"},
    {"internal": "prompt_tokens", "otel_like": "gen_ai.usage.input_tokens"},
    {"internal": "completion_tokens", "otel_like": "gen_ai.usage.output_tokens"},
    {"internal": "safety_flag", "otel_like": "gen_ai.safety.flag"},
])
mapping

,internal,otel_like
0,model_id,gen_ai.request.model
1,latency_ms,gen_ai.client.operation.duration
2,prompt_tokens,gen_ai.usage.input_tokens
3,completion_tokens,gen_ai.usage.output_tokens
4,safety_flag,gen_ai.safety.flag


## Connect to Theory
- Standard schemas reduce observability drift across tools.
- Multi-metric release gates reduce single-metric gaming.
- Trace completeness is essential for incident diagnosis.

## Frontier Case Studies & Exceptions
1. Multi-vendor LLM stack: canonical schema enabled comparable KPI dashboards.
2. RAG incident response: trace-level context identified stale retriever index.
3. Exception: provider-specific metadata still needs controlled extension fields.

## Interview Questions & Answers
1. **Q:** Why GenAI observability standards? **A:** For portability, comparability, and auditability.
2. **Q:** What is a canonical trace id used for? **A:** Correlating events across model, retrieval, and tool services.
3. **Q:** Why include token metrics? **A:** Cost and throughput management.
4. **Q:** What should release gates include? **A:** Quality, latency, safety, and cost thresholds.
5. **Q:** Why avoid single KPI gates? **A:** They can be gamed and hide regressions.
6. **Q:** What role does user feedback play? **A:** It closes the loop for evaluation and post-training prioritization.
7. **Q:** How do standards help incident response? **A:** Faster root-cause analysis from consistent telemetry fields.
8. **Q:** Why track tool-call counts? **A:** Tool behavior is a major source of agent failures and cost.
9. **Q:** Public benchmark enough? **A:** No, combine with domain-specific private eval suites.
10. **Q:** One-line guidance? **A:** Standardize telemetry early and enforce release gates with evidence.